# Build Global Multi-Region Flight Delay Dataset

This notebook builds a global dataset from US official labels and EU/Asia baseline-derived labels.

In [ ]:
import pandas as pd

from src.process_us import process_us
from src.build_baseline import build_baseline
from src.process_eu import process_eu
from src.process_asia import process_asia
from src.unify_datasets import unify_datasets


## 1) Process US

In [ ]:
# Example: provide your BTS CSV glob
US_BTS_GLOB = "data/us/raw/*.csv"
us_path = process_us(US_BTS_GLOB)
pd.read_parquet(us_path).head()


## 2) Build baseline

In [ ]:
EU_OPENSKY_ROOT = "data/parquet/flights_eu"
ASIA_OPENSKY_ROOT = "data/parquet/flights_asia"
baseline_path = build_baseline(flights_root=EU_OPENSKY_ROOT, output_path="data/processed/route_baseline.parquet")
pd.read_parquet(baseline_path).head()


## 3) Process EU

In [ ]:
eu_path = process_eu(EU_OPENSKY_ROOT, baseline_path)
pd.read_parquet(eu_path).head()


## 4) Process Asia

In [ ]:
asia_baseline_path = build_baseline(flights_root=ASIA_OPENSKY_ROOT, output_path="data/processed/route_baseline_asia.parquet")
asia_path = process_asia(ASIA_OPENSKY_ROOT, asia_baseline_path)
pd.read_parquet(asia_path).head()


## 5) Merge

In [ ]:
global_path = unify_datasets(us_path=us_path, eu_path=eu_path, asia_path=asia_path)
global_df = pd.read_parquet(global_path)
global_df.head()


## 6) Class balance table

In [ ]:
global_df.groupby(["region", "delay_label"]).size().unstack(fill_value=0)


## 7) Per-region delay percentage

In [ ]:
delay_pct = (global_df.assign(is_late=global_df['delay_label'].eq('Late'))
             .groupby('region')['is_late']
             .mean()
             .mul(100)
             .round(2))
delay_pct
